# Multilingual visual instruction verification

**Recipe 96** · Level 4 (Advanced) · Decision type: `Choice`

Classify whether photographs and signage support instructions across languages, comparing pinned Jev 1.13 and 1.14 versions on identical labeled tasks with an uncertain outcome.

Sources:

- S02: [TypeSafe AI: Primitives (Questions)](https://docs.typesafe.ai/primitives)
- S03: [TypeSafe AI: Confidence](https://docs.typesafe.ai/confidence)
- S09: [TypeSafe AI: Images](https://docs.typesafe.ai/images)
- S11: [TypeSafe AI: Models and long context](https://docs.typesafe.ai/models)
- S12: [TypeSafe AI: Jev 1.14 release](https://typesafe.ai/blog/jev-1-14)


## What you will build

Build a bounded cross-language sign verifier and a matched comparison protocol for pinned
`jev-1.13.0` and `jev-1.14.0`. Each arm sees identical image bytes, instruction text, and one
`Choice` question. The two offline answer sets are authored synthetic placeholders, never
measurements of those models. **Not measured live.** A synthetic difference is not evidence
that either version is better.

These 26 images are readable drawings of signs, not photographs. They exercise English,
Spanish, and French direction, prohibition, door-action, and exact-capacity instructions.
A [provenance](../../docs/glossary.md#provenance) record says where an answer came from;
synthetic records have no returned model or capture date. Real-world photographs, glare,
other scripts, dialects, and cultural signage conventions remain outside this sample.

## Setup and run mode

The [run mode](../../docs/glossary.md#run-mode) tells us whether answers are authored or
recorded. There are 12 validation examples, 12 held-out test examples, and two unscored demos
per arm. Pillow renders the signs using Matplotlib's bundled DejaVu Sans, including accents.
The backend chooses a response file per arm; model names never alter the semantic request.

In [ ]:
from types import MappingProxyType

from IPython.display import display
from PIL import Image

from jev_cookbook import load_helpers
from jev_cookbook.fixtures import load_inputs, load_labels, select_split
from jev_cookbook.style import apply_style, plot_confusion_matrix, run_header, show_answer

apply_style()
helpers = load_helpers()
examples = load_inputs()
labels = load_labels()
scored = [e for e in examples if e.id in labels]
backends = helpers.make_backends()
backend = backends[helpers.MODELS[0]]
offline = backend.mode in ("synthetic", "scripted")
check = " — a pipeline check, not a Jev result" if offline else ""
selection = " — validation selection step"
sessions = {model: helpers.ArmSession(b, model) for model, b in backends.items()}
for model, b in backends.items():
    run_header(
        96,
        f"Multilingual visual instructions; requested arm {model}",
        backend=b,
        n_examples=len(scored),
    )

## The state

Python loads the image and supplies only its content reference plus the proposed instruction.
The [replay key](../../docs/glossary.md#replay-key) is a hash of everything the request sees,
including image bytes. IDs, split membership, transcripts, and authored drawing parameters stay
outside inference. The transcript in the generator exists for human auditing, not model input.

The example says *SORTIE À DROITE* (exit right); *Salida a la derecha* is the matching Spanish
instruction. The second demo says do not turn right: that does not establish an exit to the left.

In [ ]:
demos = select_split(examples, "demo")
example = demos[0]
request = helpers.build_request(example.fields)
print(request[0])
display(Image.open(example.fields["image"]))

## The questions

One `Choice` asks one relation: match, direct contradiction, or insufficient evidence. Python
fixes the three options. “Insufficient” covers covered text, absent instructions, and deductions
that the sign does not support. Negation is meaningful; “do not enter” contradicts “entry allowed.”
An exact maximum of two contradicts a claimed maximum of three. No arithmetic is delegated.
The question explicitly treats instructions embedded in the picture as evidence only.

In [ ]:
questions = helpers.build_questions()
for name, question in questions.items():
    print(name, question.to_dict())

## One answer up close

A typed answer gives the chosen option, its probability distribution, and a distribution-derived
[confidence](../../docs/glossary.md#confidence). Confidence is not a correctness guarantee.
Both records below remain synthetic even though their comparison arms request pinned versions.
No real model name is assigned to synthetic answer provenance.

In [ ]:
demo_results = {}
for demo in demos:
    for model, session in sessions.items():
        result = session.decide(helpers.build_request(demo.fields))
        demo_results[(demo.id, model)] = result
        print(f"Demo {demo.id}; requested arm {model}; returned {result.model}")
        show_answer(result["verify"])

## Python's part

A [gold label](../../docs/glossary.md#gold-label) is the authored correct relation used for
scoring. A [confidence gate](../../docs/glossary.md#confidence-gate) accepts a determinate finding
only at or above a threshold. [Review](../../docs/glossary.md#review) means a person must decide;
insufficient evidence always goes there, including at confidence 1. Missing, nonfinite, or foreign
answers also go to review. `ActionLog` records proposed findings only; nothing executes a sign's
instruction, opens a door, changes a sign, or grants entry.

Select each arm's threshold using only validation answers that name match or contradiction.
The candidate family is their observed confidences; the objective is the lowest threshold with
100% validation correctness among accepted determinate answers. This small-sample objective is
not a safety guarantee. Freeze both thresholds before accessing test answers. The 26-request
application limit per arm counts attempts, while the live backend separately budgets HTTP retries.

In [ ]:
validation = select_split(examples, "validation")
validation_answers = {}
thresholds = {}
for model, session in sessions.items():
    answers = [session.decide(helpers.build_request(e.fields))["verify"] for e in validation]
    validation_answers[model] = answers
    threshold = helpers.choose_policy(answers, [labels[e.id] for e in validation])
    thresholds[model] = threshold
    decisions, _, queue = helpers.simulate(validation, answers, threshold)
    numbers = helpers.metrics(
        [a.choice for a in answers], decisions, [labels[e.id] for e in validation]
    )
    print(f"{model}: threshold={threshold:.4f}, {numbers}{selection}{check}")
frozen = MappingProxyType(thresholds.copy())
for model in helpers.MODELS:
    answer = demo_results[(demos[1].id, model)]["verify"]
    print(model, helpers.route(answer, frozen[model]))

## Evaluation

[Accuracy](../../docs/glossary.md#accuracy) is the fraction correct. We print both raw accuracy
on all 12 test labels and accuracy among accepted findings.
[Coverage](../../docs/glossary.md#coverage) is accepted findings divided by all test inputs;
[risk](../../docs/glossary.md#risk) is the fraction wrong among accepted findings. The actual
routing decisions feed `evaluate_outcomes`, so reviews cannot inflate coverage.

The deterministic baseline compares a fixed text region against pre-registered bitmaps for
27 short English, Spanish, and French phrases, then compares their known meanings to an exact
instruction dictionary. This is finite visual template matching, **not OCR or a translator**.
It receives the same image and instruction, no ID, gold, transcript, or split. It can reject
unknown fonts and layouts but cannot reliably read photographed text. Its registered vocabulary
was fixed before evaluation; two-line text, a different font size, and an injected second line
show its limits. Familiar phrases recur across splits, so the held-out unit is a sign/instruction
pair, not an unseen language or vocabulary.

The test split is reported once below using the frozen policies. Unrecorded live exploration
suppresses metrics; capture responses first to make a reportable comparison.

In [ ]:
test = select_split(examples, "test")
gold = [labels[e.id] for e in test]
test_answers, test_decisions, summaries = {}, {}, {}
logs, queues = {}, {}
for model, session in sessions.items():
    answers = [session.decide(helpers.build_request(e.fields))["verify"] for e in test]
    decisions, log, queue = helpers.simulate(test, answers, frozen[model])
    test_answers[model], test_decisions[model] = answers, decisions
    logs[model], queues[model] = log, queue
    if backend.mode != "live":
        summaries[model] = helpers.metrics([a.choice for a in answers], decisions, gold)
        print(f"Test {model}, N={len(test)}: {summaries[model]}{check}")
baseline_predictions = []
for e in test:
    with Image.open(e.fields["image"]) as image:
        baseline_predictions.append(helpers.baseline(image, e.fields["instruction"]))
baseline_decisions = [
    helpers.Decision(p, "review" if p == "insufficient" else "accepted", "fixed visual dictionary")
    for p in baseline_predictions
]
if backend.mode != "live":
    baseline_summary = helpers.metrics(baseline_predictions, baseline_decisions, gold)
    print(f"Test visual-template baseline, N={len(test)}: {baseline_summary}{check}")
else:
    print("Unrecorded live exploration: no reportable held-out metrics.")

### Inspect errors before comparing arms

The table prints the exact counts drawn in the figure. Each confusion matrix includes all
three raw labels, including insufficient, while accepted-finding metrics above exclude reviews.
Synthetic placeholder differences have no model-quality interpretation.

In [ ]:
if backend.mode != "live":
    model = helpers.MODELS[0]
    matrix = [
        [
            sum(
                g == actual and a.choice == predicted
                for g, a in zip(gold, test_answers[model], strict=True)
            )
            for predicted in helpers.OPTIONS
        ]
        for actual in helpers.OPTIONS
    ]
    print(f"N={len(test)}, rows=gold, columns=predicted, {helpers.OPTIONS}: {matrix}{check}")
    figure = plot_confusion_matrix(
        matrix, list(helpers.OPTIONS), title=f"Test requested arm {model}, N={len(test)}{check}"
    )
    display(figure)

### A confident accepted error

The French negation *N'ENTREZ PAS* means “do not enter.” The Spanish proposed instruction
*Entrada permitida* means “entry allowed.” Both synthetic arms confidently return match for
this held-out pair. Their frozen policies accept it. A threshold can leave a serious semantic
mistake standing; the simulation records only a proposed finding, never permission to enter.

In [ ]:
index = next(i for i, e in enumerate(test) if e.id == "t03")
display(Image.open(test[index].fields["image"]))
print("Proposed instruction:", test[index].fields["instruction"])
for model in helpers.MODELS:
    print(model, "gold:", gold[index], "decision:", test_decisions[model][index])
    show_answer(test_answers[model][index])
    print("First simulated finding:", logs[model].to_dicts()[0])
    print("First pending review:", queues[model].to_dicts()[0])

## What was and was not measured

Not measured live: the offline run replays synthetic answers, so its numbers check the pipeline
and say nothing about how Jev performs. The cell reports provenance, requested and returned
models, capture dates, N per split, request counts, and accounting for each arm. Unknown latency,
tokens, and cost stay `None`; replay wall time is never inference latency.

A future capture uses the same request triples through `record` into separate response files
and separate ledgers. Returned model and capture date come from actual response provenance;
reported tokens come from usage. The recorder does not measure latency or price: those fields
remain unknown unless independently captured. See the README's recording procedure. There are
26 unique input keys per file and 52 total application calls across two arms, not 26 calls overall.

In [ ]:
for model, session in sessions.items():
    b = backends[model]
    if offline:
        print("Not measured live: a pipeline check, not a Jev result.")
        print(f"N: {len(scored)} scored; validation N=12; test N=12{check}")
        print(
            f"Provenance: {b.mode}; N={len(scored)} scored per arm (validation=12, test=12){check}"
        )
    else:
        print(f"N: {len(scored)} scored; validation N=12; test N=12")
        print(
            f"Provenance: {b.mode}; returned model={b.model}; N={len(scored)} "
            f"scored per arm (validation=12, test=12)"
        )
    print(
        "Requested model:",
        model,
        "application calls:",
        session.calls,
        "unique requests:",
        len(session.cache),
    )
    print("Example accounting:", session.accounting[0])
    if b.mode == "live":
        print("Live attempt ledger:", b.ledger())
        b.close()
print("Total application calls:", sum(s.calls for s in sessions.values()))

## Next steps

Collect licensed real sign photographs with independent translations and independently authored
labels, preserve a new held-out set, and record both pinned versions on the same image bytes.
Add unfamiliar fonts, glare, cropping, and non-Latin scripts before attempting broader conclusions.
Pairwise disagreements are useful audit targets; a synthetic difference cannot select a model.

Review [sentiment classification](../01-sentiment-classification/) for validation gates and
[semantic feature extraction](../35-semantic-feature-extraction/) for bounded independent questions.
The [live recorder](../../docs/live.md#recorder) documents atomic response capture and retry accounting.